In [1]:
import re
import pandas as pd

In [2]:
dry_run_dhakouli = r"C:\Users\Nitesh\Documents\GitHubDump\ZigBee_Major_Project\testing_stuff\last_test\nitesh_ep\dry_run_dhakouli.txt"
pec_to_mohali = r"C:\Users\Nitesh\Documents\GitHubDump\ZigBee_Major_Project\testing_stuff\last_test\nitesh_ep\pec_to_mohali.txt"

pec_to_hills = r"C:\Users\Nitesh\Documents\GitHubDump\ZigBee_Major_Project\testing_stuff\last_test\nitesh_ep\pec_to_hills.txt"
hills_to_pec = r"C:\Users\Nitesh\Documents\GitHubDump\ZigBee_Major_Project\testing_stuff\last_test\nitesh_ep\hills_to_pec.txt"

files_to_process = {
    # dry_run_dhakouli : "dry_run_dhakouli",
    # pec_to_mohali : "pec_to_mohali",
    pec_to_hills : "pec_to_hills",
    hills_to_pec : "hills_to_pec"

}


pattern = re.compile(
    r'(\d{2}:\d{2}:\d{2}:\d{3}) -> <\s*([A-Z|_]+)\s*\(\s*(\d+)\s*\)\s*\|\|\s*TRANSMIT:\s*(\d+)\s*\|\|\s*\(([\d.]+),\s*([\d.]+),\s*([\d.]+)\)\s*\|\|\s*RECEIVING:\s*(\d+)\s*\|\|\s*\(([\d.]+),\s*([\d.]+),\s*([\d.]+)\)'
)


In [3]:
for file_path, file_name in files_to_process.items():
    # Read the file
    with open(file_path, 'r') as file:
        text = file.read()

    matches = pattern.findall(text)
    # print(matches[2])


    # Define column names
    columns = [
        "time", "status", "status_code", "transmit",
        "lat", "lng", "speed",
        "receiving", "foreign_lat", "foreign_lng", "foreign_speed"
    ]

    # Convert to DataFrame
    df = pd.DataFrame(matches, columns=columns)

    # Convert numeric columns from strings to float or int
    for col in df.columns:
        if col != "time" and col != "status":
            df[col] = df[col].astype(float).astype(int) if df[col].str.contains(r'^\d+$').all() else df[col].astype(float)

    df_unique = df.drop_duplicates(subset=["status_code", "lat", "lng", "speed"])

    # Optional: reset index if needed
    df_unique = df_unique.reset_index(drop=True)

    with pd.ExcelWriter(f"{file_name}.xlsx", engine="xlsxwriter") as writer:
        df.to_excel(writer, sheet_name="whole", index=False)
        df_unique.to_excel(writer, sheet_name="unique", index=False)


